# Phase 7A: Temporal Robustness and Model Aging

This notebook is a presentation layer. It reads the generated Phase 7A artifacts; training and cascade logic remain under `src/`.

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
OUT = ROOT / 'results' / 'experiments' / 'phase7a'
results = json.loads((OUT / 'phase7a_results.json').read_text(encoding='utf-8'))
periods = pd.DataFrame(results['period_definitions'])
metrics = pd.read_csv(OUT / 'phase7a_period_metrics.csv')
predictions = pd.read_csv(OUT / 'phase7a_predictions.csv')
routing = pd.read_csv(OUT / 'phase7a_routing_diagnostics.csv')

## Period definitions

In [ ]:
periods

## Forward-only metrics

In [ ]:
metric_columns = ['prediction_period', 'precision', 'recall', 'f1', 'pr_auc', 'roc_auc', 'balanced_accuracy', 'tn', 'fp', 'fn', 'tp']
metrics[metric_columns]

In [ ]:
metrics.set_index('prediction_period')[['precision', 'recall', 'f1', 'pr_auc', 'balanced_accuracy']].plot(
    marker='o', ylim=(0, 1), title='Frozen MulCas performance by prediction period'
)

## Routing diagnostics

In [ ]:
predictions.groupby(['period', 'accepted_viewer']).size().unstack(fill_value=0)

Metric changes are descriptive temporal variation. They do not by themselves prove concept drift, and no Phase 7A result is used to alter the frozen model.

# Phase 7B: Frozen MulCas component/view ablation

These tables present predeclared ablations only. They are not model-selection results.

In [ ]:
OUT_7B = ROOT / 'results' / 'experiments' / 'phase7b'
ablation_metrics = pd.read_csv(OUT_7B / 'phase7b_ablation_metrics.csv')
ablation_deltas = pd.read_csv(OUT_7B / 'phase7b_metric_deltas.csv')
ablation_routing = pd.read_csv(OUT_7B / 'phase7b_routing.csv')
frozen_viewers = json.loads((OUT_7B / 'phase7b_frozen_viewers.json').read_text(encoding='utf-8'))

## Actual frozen viewer sequence

In [ ]:
pd.DataFrame(frozen_viewers['viewers'])

## Ablation metrics and deltas

In [ ]:
ablation_metrics[['ablation_id', 'prediction_period', 'precision', 'recall', 'f1', 'pr_auc', 'balanced_accuracy', 'tn', 'fp', 'fn', 'tp']]

In [ ]:
ablation_deltas.pivot(index='ablation_id', columns='prediction_period', values='delta_f1')

In [ ]:
ablation_routing.pivot_table(index=['ablation_id', 'prediction_period'], columns='route', values='count', fill_value=0)

# Phase 7C: Chronological creator-feature ablation

The comparison changes only whether the leakage-safe creator column is present.

In [ ]:
OUT_7C = ROOT / 'results' / 'experiments' / 'phase7c'
creator_metrics = pd.read_csv(OUT_7C / 'phase7c_period_metrics.csv')
creator_deltas = pd.read_csv(OUT_7C / 'phase7c_metric_deltas.csv')
creator_diagnostics = pd.read_csv(OUT_7C / 'phase7c_creator_diagnostics.csv')
creator_results = json.loads((OUT_7C / 'phase7c_results.json').read_text(encoding='utf-8'))

## With versus without creator history

In [ ]:
creator_metrics[['prediction_period', 'condition', 'precision', 'recall', 'f1', 'pr_auc', 'balanced_accuracy', 'tn', 'fp', 'fn', 'tp']]

In [ ]:
creator_deltas[['prediction_period', 'delta_precision', 'delta_recall', 'delta_f1', 'delta_pr_auc', 'delta_balanced_accuracy', 'delta_fp', 'delta_fn']]

## Creator-history prevalence and error groups

In [ ]:
creator_diagnostics

In [ ]:
pd.DataFrame([
    {'period': row['prediction_period'], 'viewer': row['viewer'], **{f'delta_{k}': v for k, v in row['deltas'].items()}}
    for row in creator_results['viewer_level_diagnostics']
])[['period', 'viewer', 'delta_precision', 'delta_recall', 'delta_f1', 'delta_pr_auc', 'delta_fp', 'delta_fn']]

# Phase 7D: Systematic error analysis

This section reads the exact frozen-prediction diagnostics generated by `src.experiments.phase7d`. It is descriptive only and performs no training or tuning.

In [ ]:
OUT_7D = ROOT / 'results' / 'experiments' / 'phase7d'
period_errors = pd.read_csv(OUT_7D / 'phase7d_period_summary.csv')
routing_errors = pd.read_csv(OUT_7D / 'phase7d_routing_errors.csv')
clone_diagnostics = pd.read_csv(OUT_7D / 'phase7d_clone_diagnostics.csv')
p5_analysis = pd.read_csv(OUT_7D / 'phase7d_p5_analysis.csv')
phase7d_results = json.loads((OUT_7D / 'phase7d_results.json').read_text(encoding='utf-8'))
period_errors

## Confusion inventory and routing

In [ ]:
routing_errors.pivot(index='routing_error_category', columns='period', values='count').fillna(0).astype(int)

## Historical clone prevalence

In [ ]:
clone_diagnostics[clone_diagnostics['error_type'].isin(['false_positive', 'false_negative', 'true_positive'])]

## P5 false negatives versus true positives

In [ ]:
p5_analysis[(p5_analysis['statistic'] == 'mean') & p5_analysis['feature'].isin(['previously_seen_creator', 'creator_history_feature', 'opcode_count', 'unique_opcode_count', 'opcode_entropy', 'parseable_operand_count', 'operand_width_mean', 'prediction_confidence', 'reached_voting_stage'])].pivot(index='feature', columns='error_type', values='value')